In [1]:
import warnings
warnings.filterwarnings('ignore')
import polars as pl
import polars.selectors as cs
import numpy as np
import math
import seaborn as sns
import os 
import re
import matplotlib.pyplot as plt
import gzip
import matplotlib.colors as mcolors
from scipy import stats
pl.Config.set_fmt_str_lengths(50)
pl.Config().set_tbl_rows(50)
sns.set_style(style='white')
warnings.filterwarnings('ignore')

1. Validate that the luciferase assay recapitulates ccMPRA-identified activity and provides baseline measurements for each promoter.
-- 20 strong enhancers (10 encode and 10 non-encode annotated) + the promoters alone
-- 20 strong silencers + the promoters alone
= 80 sequences
2. Test the promoter-dependent activity.
- 3 CREs that act as enhancers or silencers based on the promoter + the promoters alone
- 3 CREs that act as enhancer for 1 promoter and no effect on the other promoter + the promoters alone
- 3 CREs that act as silencer for 1 promoter and no effect on the other promoter + the promoters alone
= 36 sequences
3. Then the reviewer 2 was worried about the effect of the length variation, coordinates variation and distance from H3k27ac peaks so I was thinking:
- Reuse 6 strong enhancers and 6 strong silencers that we tested in point 1 and test 2 additional lengths for them (20). No need to redo the promoter alone. Select 3 bins that are highly covered with sequences and 3 that are lowly covered. Then select three different sequences with different lengths that were in the same bin for highly covered, and just the entire sequence for the lowly covered bins.
- Among the promoters we used in point 1, we select 3 enhancers and 3 silencers, and we try an other length for them, either alone either with their CRE (fixed length, same as point 1) (10)
2 of each with high coverage bins, and 1 of each with low coverage bins.
 
= 40 sequences

## Filtering

Filtering on:
- both parts at least 100 bp
- CRE is not an encode PLS
- promoter contains TSS


In [3]:
cd = "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/"
data = pl.read_csv(os.path.join(cd,"results/MPRA_analysis/CMPRA5/labeled_data_promoteroa_OA.tsv"), separator="\t")
data = data.filter(pl.col("right_bin").str.contains("null").not_()).rename({"OE": "CRE", "nr_reads": "nr_barcodes", "dist": "distance", "interaction": "ENCODE_labels"})
data = data.filter(pl.col("label") != "other - other")
data = data.filter(pl.col("any_tss") == "yes")
data = data.with_columns(
	CRE_length = pl.col("CRE").str.split("-").list.get(2).cast(pl.Int64) - pl.col("CRE").str.split("-").list.get(1).cast(pl.Int64),
	promoter_length = pl.col("promoter").str.split("-").list.get(2).cast(pl.Int64) - pl.col("promoter").str.split("-").list.get(1).cast(pl.Int64)
)
data = data.filter((pl.col("CRE_length") >= 100) & (pl.col("promoter_length") >= 100))
data = data.with_columns(pl.when(pl.all_horizontal(pl.any_horizontal(cs.matches("screen").str.contains("PLS")) & pl.any_horizontal(cs.matches("screen").is_null())))
			 			.then(pl.lit("PLS - undefined"))
						.when(pl.all_horizontal(pl.any_horizontal(cs.matches("screen").str.contains("ELS")) & pl.any_horizontal(cs.matches("screen").is_null())))
						.then(pl.lit("ELS - undefined"))
						 .when(pl.all_horizontal(cs.matches("screen").is_null())).then(pl.lit("undefined")).otherwise(pl.col("ENCODE_labels"))
						 .alias("ENCODE_labels"))
data = data.filter(pl.col("ENCODE_labels").str.contains("PLS - PLS").not_()).select(~cs.matches("left|right|tss|Val|std"))

## Enhancers and silencers

In [21]:
silencers = pl.concat([data.filter(pl.col("ENCODE_labels") == "PLS - undefined").sort("z_score").head(200),
						data.filter(pl.col("ENCODE_labels") == "PLS - ELS").sort("z_score").head(200)])
enhancers = pl.concat([data.filter(pl.col("ENCODE_labels") == "PLS - undefined").sort("z_score", descending=True).head(200),
						data.filter(pl.col("ENCODE_labels") == "PLS - ELS").sort("z_score", descending=True).head(200)])

## Multi interacting CREs

In [45]:
multi_prom = data.filter(pl.col("promoter").n_unique().over("CRE") >= 2) \
	.filter((pl.col("z_score").max().over("CRE") > 1.5) | (pl.col("z_score").min().over("CRE") < -1.5)) \
	.with_columns(activity_difference = np.abs(pl.col("z_score").max().over("CRE") - pl.col("z_score").min().over("CRE")))\
		.sort("activity_difference", descending=True)

multi_prom = multi_prom.filter(pl.col("CRE").is_in(multi_prom.select("CRE").unique(maintain_order=True).head(100)["CRE"]))

## Selecting the dual function CREs

In [50]:
multi_prom.filter(pl.col("target_genes") == "BAZ1A")
multi_prom.filter(pl.col("CRE") == "chr14-34877605-34877875-.")

logFC,nr_barcodes,nr_seqs,label,ENCODE_labels,distance,target_genes,effect,promoter,CRE,promoter_only,z_score,CRE_length,promoter_length,activity_difference
f64,i64,i64,str,str,f64,str,str,str,str,f64,f64,i64,i64,f64
-1.210021,12,4,"""target - other""","""PLS - undefined""",1806.0,"""BAZ1A""","""no effect""","""chr14-34875554-34876314--""","""chr14-34877605-34877875-.""",-0.613998,-0.997614,270,760,2.352574
-1.134578,36,10,"""target - other""","""PLS - undefined""",2293.5,"""BAZ1A""","""no effect""","""chr14-34875340-34875553--""","""chr14-34877605-34877875-.""",-0.471866,-1.061715,270,213,2.352574
-1.527458,5,1,"""target - other""","""PLS - undefined""",244752.0,"""PPP2R3C""","""downregulating""","""chr14-35122081-35122903--""","""chr14-34877605-34877875-.""",-0.224263,-3.350188,270,822,2.352574


In [ ]:
dual_function = multi_prom.sort("activity_difference", descending=True)\
.filter((pl.col("z_score").min().over("CRE") < -1.2) & (pl.col("z_score").max().over("CRE") > 1.2)\
& (pl.col("promoter") != "chr17-75261786-75262174--")
& (pl.col("promoter").str.contains("chr20-17968096-17968866").not_())
& (pl.col("CRE") != "chr14-23017677-23018228-."))
dual_function = dual_function.filter(pl.col("CRE").is_in(dual_function.select("CRE").unique(maintain_order=True).head(3)["CRE"]))
dual_function

logFC,nr_barcodes,nr_seqs,label,ENCODE_labels,distance,target_genes,effect,promoter,CRE,promoter_only,z_score,CRE_length,promoter_length,activity_difference
f64,i64,i64,str,str,f64,str,str,str,str,f64,f64,i64,i64,f64
1.227543,5,2,"""target - other""","""PLS - undefined""",8630.0,"""GGA3""","""no effect""","""chr17-75261404-75261785--""","""chr17-75252859-75253070-.""",0.602591,1.722083,211,381,4.307408
-0.200868,11,3,"""target - other""","""PLS - undefined""",9015.5,"""MRPS7""","""no effect""","""chr17-75261786-75262174-+""","""chr17-75252859-75253070-.""",0.440545,-1.836872,211,388,4.307408
1.044954,7,2,"""target - other""","""PLS - ELS""",255839.5,"""TOMM6""","""no effect""","""chr6-41787436-41787975-+""","""chr6-42043482-42043608-.""",0.463298,1.87923,126,539,3.529796
-0.94513,5,2,"""target - other""","""PLS - ELS""",122455.5,"""BYSL""","""no effect""","""chr6-41920722-41921457-+""","""chr6-42043482-42043608-.""",0.237817,-1.650566,126,735,3.529796
-0.932809,9,1,"""target - other""","""PLS - undefined""",163130.0,"""PPP2R3C""","""no effect""","""chr14-35122081-35122903--""","""chr14-34959309-34959415-.""",-0.224263,-1.821493,106,822,3.30441
0.453758,14,2,"""target - other""","""PLS - undefined""",83915.5,"""BAZ1A""","""no effect""","""chr14-34875340-34875553--""","""chr14-34959309-34959415-.""",-0.471866,1.482918,106,213,3.30441


In [54]:
enhancing_or_not = multi_prom.sort("activity_difference", descending=True)\
.filter((pl.col("z_score").max().over("CRE") > 1.5) & (pl.col("z_score").min().over("CRE") < 1)
& (pl.col("z_score").min().over("CRE") > -0.5)
& (pl.col("CRE") != "chr14-20955217-20955500-.")
& (pl.col("CRE") != "chr16-46708558-46708713-."))
enhancing_or_not = enhancing_or_not.filter(pl.col("CRE").is_in(enhancing_or_not.select("CRE").unique(maintain_order=True).head(3)["CRE"]))
enhancing_or_not

logFC,nr_barcodes,nr_seqs,label,ENCODE_labels,distance,target_genes,effect,promoter,CRE,promoter_only,z_score,CRE_length,promoter_length,activity_difference
f64,i64,i64,str,str,f64,str,str,str,str,f64,f64,i64,i64,f64
0.430336,13,1,"""negative - other""","""undefined""",166242.5,"""WFDC9""","""upregulating""","""chr20-45631225-45631710--""","""chr20-45465166-45465284-.""",-0.85562,3.611017,118,485,3.417972
0.345329,8,1,"""positive - other""","""PLS - undefined""",49139.0,"""PIGT""","""no effect""","""chr20-45415817-45416355-+""","""chr20-45465166-45465284-.""",0.289312,0.193045,118,538,3.417972
-0.483092,5,1,"""positive - other""","""PLS - ELS""",66240.5,"""PDF""","""no effect""","""chr16-69330001-69330668--""","""chr16-69396489-69396661-.""",-0.687894,0.479575,172,667,2.781948
1.69413,8,1,"""positive - other""","""PLS - ELS""",28068.5,"""CYB5B""","""upregulating""","""chr16-69424473-69424814-+""","""chr16-69396489-69396661-.""",-0.01768,3.261523,172,341,2.781948
-0.607768,5,1,"""target - other""","""PLS - ELS""",291038.0,"""RNASE4""","""no effect""","""chr14-20683835-20684397-+""","""chr14-20975098-20975210-.""",-0.917317,0.973539,112,562,2.563649
0.069269,6,1,"""negative - other""","""PLS - ELS""",83863.5,"""RNASE3""","""upregulating""","""chr14-20891166-20891415-+""","""chr14-20975098-20975210-.""",-1.288813,3.537188,112,249,2.563649


In [57]:
silencing_or_not = multi_prom.sort("activity_difference", descending=True)\
.filter((pl.col("z_score").min().over("CRE") < -1.5) & (pl.col("z_score").max().over("CRE") > -1)
& (pl.col("z_score").max().over("CRE") < 1) & 
(pl.col("target_genes").n_unique().over("CRE") > 1) &
( (pl.col("CRE") == "chr14-35441982-35442128-.")
| (pl.col("CRE") == "chr6-7534112-7534296-.")
| (pl.col("CRE") == "chr2-74152117-74152296-.")
& (pl.col("promoter") != "chr2-73829060-73829209-+")
))
silencing_or_not = silencing_or_not.filter(pl.col("CRE").is_in(silencing_or_not.select("CRE").unique(maintain_order=True).head(230)["CRE"]))
silencing_or_not

logFC,nr_barcodes,nr_seqs,label,ENCODE_labels,distance,target_genes,effect,promoter,CRE,promoter_only,z_score,CRE_length,promoter_length,activity_difference
f64,i64,i64,str,str,f64,str,str,str,str,f64,f64,i64,i64,f64
0.535836,7,2,"""target - other""","""PLS - undefined""",4098.0,"""BOLA3""","""no effect""","""chr2-74147827-74148390--""","""chr2-74152117-74152296-.""",0.100867,0.939548,179,563,2.904546
-0.172381,5,1,"""positive - other""","""PLS - undefined""",323340.0,"""STAMBP""","""no effect""","""chr2-73828674-73829059-+""","""chr2-74152117-74152296-.""",0.911029,-1.814256,179,385,2.904546
0.222777,6,1,"""target - other""","""PLS - undefined""",37266.5,"""NFKBIA""","""no effect""","""chr14-35404489-35405088--""","""chr14-35441982-35442128-.""",0.171183,0.173374,146,599,2.595098
-1.166294,6,1,"""target - other""","""PLS - undefined""",319563.0,"""PPP2R3C""","""downregulating""","""chr14-35122081-35122903--""","""chr14-35441982-35442128-.""",-0.224263,-2.421725,146,822,2.595098
-0.499211,8,1,"""target - other""","""PLS - undefined""",144641.5,"""RIOK1""","""downregulating""","""chr6-7389289-7389836-+""","""chr6-7534112-7534296-.""",0.77628,-2.122709,184,547,2.002511
-0.50902,5,2,"""target - other""","""PLS - undefined""",7377.5,"""DSP""","""no effect""","""chr6-7541219-7541944-+""","""chr6-7534112-7534296-.""",-0.46161,-0.120198,184,725,2.002511


In [58]:
all_dual = pl.concat([dual_function, enhancing_or_not, silencing_or_not])
all_dual.select("CRE").unique().height

9

## Getting bin order of original sequences for enhancers and silencers

In [5]:
!zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/mprasnakeflow/results/assigned_bcs_part1and2.tsv.gz' | cut -f -2,4,6,8,10,12,14 | awk 'NR > 1 && NF >=5' | cut -f 2 > temp.seqids.tsv

In [90]:
pl.concat([silencers, enhancers]).select(pl.col("CRE")).write_csv("CREs.temp.ids.tsv", include_header=False)
pl.concat([silencers, enhancers]).select(pl.col("promoter")).write_csv("promoter.temp.ids.tsv", include_header=False)

In [7]:
seq_ids = "temp.seqids.tsv"
cre_ids = "CREs.temp.ids.tsv"
prom_ids = "promoter.temp.ids.tsv"

In [97]:
!awk -v FS="\t" '{print $1"-"$2"-"$3"-"$4"S"$5"-"$6"-"$7"-"$8"\t"$9}' \
	<(zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/promoteroabinaware_bins_all.filt_OA.bed.gz') \
		| grep -v "\-\-\-" | grep -Fwf <(paste -d "S" CREs.temp.ids.tsv promoter.temp.ids.tsv) | grep -Fwf <(sed 's/>//' temp.seqids.tsv) \
			| sed 's/S/\t/' > temp.actualbins.tsv

In [98]:
# Check uniqueness
! echo $(cut -f -2 temp.actualbins.tsv | sort | uniq |  wc -l) $(wc -l < temp.actualbins.tsv)
! echo $(wc -l < CREs.temp.ids.tsv) # looks like there are some incorrect bins

494 494


800


In [22]:
actual_bins = pl.read_csv("temp.actualbins.tsv", separator="\t", has_header=False).rename({"column_1": "left_bin", "column_2": "right_bin"}) 

In [11]:
actual_bins.with_columns(column_3 = pl.col("column_3").str.split(",").list.len()).head()

left_bin,right_bin,column_3
str,str,u32
"""chr1-11805849-11806206-+""","""chr1-11842196-11842568-.""",12
"""chr1-12392815-12393089-.""","""chr1-12618206-12618597--""",14
"""chr1-12393090-12393210-.""","""chr1-12618206-12618597--""",13
"""chr1-12618206-12618597--""","""chr1-12350334-12350578-.""",17
"""chr1-12618206-12618597--""","""chr1-12565474-12565661-.""",33


In [102]:
! grep "chr20-35768547-35768902-." temp.actualbins.tsv

chr20-35768547-35768902-.	chr20-35742180-35742625--	m84066_240516_024507_s2/120980222/ccs,m84066_240516_024507_s2/128848589/ccs,m84066_240516_024507_s2/188548809/ccs,m84066_240516_024507_s2/224855237/ccs,m84066_240516_024507_s2/44960585/ccs,m84066_240516_024507_s2/99552466/ccs,m84066_240623_102831_s3/130484144/ccs,m84066_240623_102831_s3/252512281/ccs,m84066_240623_122801_s1/152966447/ccs,m84066_240623_122801_s1/168168193/ccs,m84066_240623_122801_s1/19599883/ccs,m84066_240623_122801_s1/227476570/ccs,m84066_240623_122801_s1/239863727/ccs,m84066_240623_122801_s1/25105034/ccs,m84066_240623_122801_s1/267258915/ccs,m84066_240623_122801_s1/43583955/ccs,m84066_240623_122801_s1/4656783/ccs,m84066_240623_122801_s1/49154991/ccs,m84066_240623_122801_s1/59774346/ccs,m84066_240623_122801_s1/62461404/ccs,m84066_240623_122801_s1/63115199/ccs,m84066_240623_122801_s1/72750227/ccs,V350161981_L03_contig_2531219


In [13]:
# --------- Code if we want to try multiple bin orders -----------

# Get the silencers with the bin order in which they were sequenced, and additionally in a CRE - promoter order
silencers_with_bin_order = silencers.select(pl.col("CRE").alias("left_bin"), pl.col("promoter").alias("right_bin"), pl.all())
silencers_with_bin_order = pl.concat([silencers_with_bin_order, 
									  actual_bins.select(pl.exclude("column_3")).join(silencers, left_on=["left_bin", "right_bin"], right_on=["promoter", "CRE"], coalesce = False)])

enhancers_with_bin_order = enhancers.select(pl.col("CRE").alias("left_bin"), pl.col("promoter").alias("right_bin"), pl.all())
enhancers_with_bin_order = pl.concat([enhancers_with_bin_order, 
									  actual_bins.select(pl.exclude("column_3")).join(enhancers, left_on=["left_bin", "right_bin"], right_on=["promoter", "CRE"], coalesce = False)])
enhancers_with_bin_order.height


630

In [23]:
# --------- Code if we want to test CRE - promoter order only -----------

silencers_with_bin_order = actual_bins.select(pl.exclude("column_3")).join(silencers, left_on=["left_bin", "right_bin"], right_on=["CRE", "promoter"], coalesce = False)

enhancers_with_bin_order = actual_bins.select(pl.exclude("column_3")).join(enhancers, left_on=["left_bin", "right_bin"], right_on=["CRE", "promoter"], coalesce = False)
silencers_with_bin_order.height, silencers.height

(257, 400)

In [101]:
silencers_with_bin_order.filter(pl.col("CRE") =="chr20-35768547-35768902-.")

left_bin,right_bin,logFC,nr_barcodes,nr_seqs,label,ENCODE_labels,distance,target_genes,effect,promoter,CRE,promoter_only,z_score,CRE_length,promoter_length
str,str,f64,i64,i64,str,str,f64,str,str,str,str,f64,f64,i64,i64
"""chr20-35768547-35768902-.""","""chr20-35742180-35742625--""",1.125667,9,3,"""target - other""","""PLS - ELS""",26322.0,"""RBM39""","""downregulating""","""chr20-35742180-35742625--""","""chr20-35768547-35768902-.""",2.328104,-3.129785,355,445


In [15]:
silencers_with_bin_order.filter(pl.col("ENCODE_labels") == "PLS - undefined").height, enhancers_with_bin_order.filter(pl.col("ENCODE_labels") == "PLS - undefined").height
#silencers_with_bin_order.filter(pl.col("ENCODE_labels") == "PLS - ELS").height, enhancers_with_bin_order.filter(pl.col("ENCODE_labels") == "PLS - ELS").height

(125, 113)

## Getting bin order of original sequences for multi-promoter CREs

In [59]:
all_dual.select(pl.col("CRE")).write_csv("CREs.multiprom.temp.ids.tsv", include_header=False)
all_dual.select(pl.col("promoter")).write_csv("promoter.multiprom.temp.ids.tsv", include_header=False)
cre_ids = "CREs.multiprom.temp.ids.tsv"
prom_ids = "promoter.multiprom.temp.ids.tsv"

In [60]:
!awk -v FS="\t" '{print $1"-"$2"-"$3"-"$4"\t"$5"-"$6"-"$7"-"$8"\t"$9}' \
	<(zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/promoteroabinaware_bins_all.filt_OA.bed.gz') \
		| grep -Fwf <(paste CREs.multiprom.temp.ids.tsv  promoter.multiprom.temp.ids.tsv) | grep -Fwf <(sed 's/>//' temp.seqids.tsv)  \
			> temp.multiprom1.actualbins.tsv

In [61]:
!awk -v FS="\t" '{print $1"-"$2"-"$3"-"$4"\t"$5"-"$6"-"$7"-"$8"\t"$9}' \
	<(zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/promoteroabinaware_bins_all.filt_OA.bed.gz') \
		| grep -Fwf <(paste promoter.multiprom.temp.ids.tsv CREs.multiprom.temp.ids.tsv ) | grep -Fwf <(sed 's/>//' temp.seqids.tsv) \
			> temp.multiprom2.actualbins.tsv

In [62]:
!wc -l CREs.multiprom.temp.ids.tsv

18 CREs.multiprom.temp.ids.tsv


In [63]:
# Check bin orders for each of the sequences that were used in the dual function bins
!cat temp.multiprom1.actualbins.tsv temp.multiprom2.actualbins.tsv| awk '{split($3,seqs,","); for (i in seqs) print $1"\t"$2"\t"seqs[i]}'| sort | uniq -c | awk '{print $2"\t"$3"\t"$4"\t"$5"\t"$1}' | grep -Fwf <(sed 's/>//' temp.seqids.tsv) \
	|cut -f -2 | sort | uniq -c 

      1 chr14-20683835-20684397-+	chr14-20975098-20975210-.
      1 chr14-20891166-20891415-+	chr14-20975098-20975210-.
      1 chr14-34875340-34875553--	chr14-34959309-34959415-.
      1 chr14-34959309-34959415-.	chr14-34875340-34875553--
      1 chr14-35122081-35122903--	chr14-34959309-34959415-.
      1 chr14-35441982-35442128-.	chr14-35122081-35122903--
      1 chr14-35441982-35442128-.	chr14-35404489-35405088--
      1 chr16-69330001-69330668--	chr16-69396489-69396661-.
      1 chr16-69396489-69396661-.	chr16-69424473-69424814-+
      1 chr17-75252859-75253070-.	chr17-75261404-75261785--
      2 chr17-75252859-75253070-.	chr17-75261786-75262174-+
      1 chr17-75261404-75261785--	chr17-75252859-75253070-.
      1 chr17-75261786-75262174-+	chr17-75252859-75253070-.
      1 chr2-73828674-73829059-+	chr2-74152117-74152296-.
      2 chr2-74152117-74152296-.	chr2-74147827-74148390--
      1 chr20-45465166-45465284-.	chr20-45415817-45416355-+
      1 chr20-45631225-45631710--	chr20-4546

In [64]:
# Checking the original bin order for those sequences that don't appear in both bin orders
!cat temp.multiprom1.actualbins.tsv temp.multiprom2.actualbins.tsv | \
	awk '{if ($1 < $2) {a[$1$2]++; lines[$1$2] = $0;} \
		else {a[$2$1]++; lines[$2$1] = $0;}} END {for (i in a) if(a[i] == 1) print lines[i]}' 

chr14-35441982-35442128-.	chr14-35122081-35122903--	m84066_240516_024507_s2/119734995/ccs,m84066_240516_024507_s2/131928464/ccs,m84066_240516_024507_s2/174065101/ccs,m84066_240516_024507_s2/211943887/ccs,m84066_240516_024507_s2/216401761/ccs,m84066_240623_102831_s3/139922033/ccs,m84066_240623_102831_s3/159319464/ccs,m84066_240623_102831_s3/161481633/ccs,m84066_240623_102831_s3/176489976/ccs,m84066_240623_102831_s3/219156613/ccs,m84066_240623_102831_s3/222303961/ccs,m84066_240623_102831_s3/248647690/ccs,m84066_240623_122801_s1/20907489/ccs,m84066_240623_122801_s1/227607902/ccs,m84066_240623_122801_s1/243927472/ccs,m84066_240623_122801_s1/244585077/ccs,m84066_240623_122801_s1/32576346/ccs,m84066_240623_122801_s1/38798991/ccs
chr20-45631225-45631710--	chr20-45465166-45465284-.	m84066_240623_102831_s3/25496946/ccs,m84066_240623_102831_s3/28380396/ccs,m84066_240623_102831_s3/47911624/ccs,m84066_240623_102831_s3/52560763/ccs,m84066_240623_102831_s3/57282515/ccs,m84066_240623_122801_s1/125567

In [65]:
# Checking how many have the same CRE
!cat temp.multiprom1.actualbins.tsv temp.multiprom2.actualbins.tsv | \
	awk '{if ($1 < $2) {a[$1$2]++; lines[$1$2] = $0;} \
		else {a[$2$1]++; lines[$2$1] = $0;}} END {for (i in a) if(a[i] == 1) print lines[i]}' \
			| cut -f 1 | sort | uniq -c

      1 chr14-20683835-20684397-+
      1 chr14-20891166-20891415-+
      1 chr14-35122081-35122903--
      2 chr14-35441982-35442128-.
      1 chr16-69330001-69330668--
      1 chr16-69396489-69396661-.
      1 chr2-73828674-73829059-+
      1 chr2-74152117-74152296-.
      1 chr20-45465166-45465284-.
      1 chr20-45631225-45631710--
      1 chr6-41787436-41787975-+
      1 chr6-42043482-42043608-.
      1 chr6-7389289-7389836-+


In [66]:
# Checking how many have the same CRE
!cat temp.multiprom1.actualbins.tsv temp.multiprom2.actualbins.tsv | \
	awk '{if ($1 < $2) {a[$1$2]++; lines[$1$2] = $0;} \
		else {a[$2$1]++; lines[$2$1] = $0;}} END {for (i in a) if(a[i] == 1) print lines[i]}' \
			| cut -f 2 | sort | uniq -c

      2 chr14-20975098-20975210-.
      1 chr14-34959309-34959415-.
      1 chr14-35122081-35122903--
      1 chr14-35404489-35405088--
      1 chr16-69396489-69396661-.
      1 chr16-69424473-69424814-+
      1 chr2-74147827-74148390--
      1 chr2-74152117-74152296-.
      1 chr20-45415817-45416355-+
      1 chr20-45465166-45465284-.
      1 chr6-41920722-41921457-+
      1 chr6-42043482-42043608-.
      1 chr6-7534112-7534296-.


In [16]:
# Checking whether those in both directions have the same CRE
!cat temp.multiprom1.actualbins.tsv temp.multiprom2.actualbins.tsv | \
	awk '{if ($1 < $2) {a[$1$2]++; lines[$1$2] = $0;} \
		else {a[$2$1]++; lines[$2$1] = $0;}} END {for (i in a) if(a[i] > 1) print lines[i]}' 

chr6-7541219-7541944-+	chr6-7534112-7534296-.	m84066_240516_024507_s2/132121590/ccs,m84066_240516_024507_s2/150804324/ccs,m84066_240623_102831_s3/136841101/ccs,m84066_240623_102831_s3/198972741/ccs,m84066_240623_102831_s3/35460115/ccs,m84066_240623_102831_s3/78515178/ccs,m84066_240623_102831_s3/9504604/ccs,m84066_240623_122801_s1/103813699/ccs,m84066_240623_122801_s1/108334667/ccs,m84066_240623_122801_s1/114037017/ccs,m84066_240623_122801_s1/167448978/ccs,m84066_240623_122801_s1/220660968/ccs,m84066_240623_122801_s1/87492332/ccs,V350161981_L02_contig_1210620,V350161981_L02_contig_2693511
chr17-75261404-75261785--	chr17-75252859-75253070-.	m84066_240516_024507_s2/105382920/ccs,m84066_240516_024507_s2/136709011/ccs,m84066_240516_024507_s2/165021796/ccs,m84066_240516_024507_s2/233641454/ccs,m84066_240516_024507_s2/258215774/ccs,m84066_240516_024507_s2/33491721/ccs,m84066_240516_024507_s2/53810769/ccs,m84066_240623_102831_s3/173933315/ccs,m84066_240623_102831_s3/238880969/ccs,m84066_240623

In [58]:
mpra = pl.read_csv(os.path.join(cd,"results/MPRA_analysis/CMPRA5/labeled_data_promoteroa_binaware_OA.tsv"),
				   separator="\t")

In [69]:
mpra.filter(pl.col("promoter").count().over("OE") > 2).sort("OE").head()

logFC,adj.P.Val,P.Value,left_bin,right_bin,nr_reads,nr_seqs,any_tss,screen_left,screen_right,targeted_left,targeted_right,target_gene_left,target_gene_right,label,interaction,dist,target_genes,effect,promoter,OE,promoter_only,std,z_score
f64,f64,f64,str,str,i64,i64,str,str,str,str,str,str,str,str,str,f64,str,str,str,str,f64,f64,f64
-1.210144,6.0737e-7,2.2907e-7,"""chr1-103613718-103613899-.""","""chr1-103617113-103617668-+""",6,1,"""yes""",null,null,"""unlabeled""","""negativecontrol""","""unlabeled""","""AMY2A""","""negative - other""",null,3582.0,"""AMY2A""","""no effect""","""chr1-103617113-103617668-+""","""chr1-103613718-103613899-.""",-1.198857,0.324113,-0.034824
-1.348953,5.1786e-8,1.6375e-8,"""chr1-103613718-103613899-.""","""chr1-103616844-103617112-+""",6,1,"""no""",null,"""pELS,CTCF-bound""","""unlabeled""","""negativecontrol""","""unlabeled""","""AMY2A""","""negative - other""",null,3169.5,"""AMY2A""","""no effect""","""chr1-103616844-103617112-+""","""chr1-103613718-103613899-.""",-1.210347,0.290864,-0.476531
-1.01389,0.000014,0.000007,"""chr1-103617113-103617668-+""","""chr1-103613718-103613899-.""",8,3,"""yes""",null,null,"""negativecontrol""","""unlabeled""","""AMY2A""","""unlabeled""","""negative - other""",null,3582.0,"""AMY2A""","""no effect""","""chr1-103617113-103617668-+""","""chr1-103613718-103613899-.""",-1.198857,0.324113,0.570685
0.506356,0.000023,0.000011,"""chr1-112606732-112606874-.""","""chr1-112619436-112619859-+""",16,2,"""yes""",null,"""PLS,CTCF-bound""","""unlabeled""","""target""","""unlabeled""","""CAPZA1""","""target - other""",null,12844.5,"""CAPZA1""","""no effect""","""chr1-112619436-112619859-+""","""chr1-112606732-112606874-.""",0.872174,0.616977,-0.59292
0.596705,0.002973,0.001913,"""chr1-112606732-112606874-.""","""chr1-112618940-112619435-+""",5,1,"""yes""",null,"""PLS,CTCF-bound""","""unlabeled""","""target""","""unlabeled""","""CAPZA1""","""target - other""",null,12384.5,"""CAPZA1""","""no effect""","""chr1-112618940-112619435-+""","""chr1-112606732-112606874-.""",0.137157,0.337682,1.360889


In [65]:
actual_bins = pl.read_csv("temp.multiprom.actualbins.tsv", separator="\t", has_header=False).rename({"column_1": "left_bin", "column_2": "right_bin"})

In [34]:
actual_bins.with_columns(column_3 = pl.col("column_3").str.split(",").list.len()).head()

left_bin,right_bin,column_3
str,str,u32
"""chr14-20688098-20688455-+""","""chr14-20955217-20955500-.""",16
"""chr14-20955217-20955500-.""","""chr14-20890827-20891165-+""",16
"""chr14-35115858-35116161-.""","""chr14-35121748-35122080--""",70
"""chr14-35121748-35122080--""","""chr14-35115858-35116161-.""",67
"""chr14-35122081-35122903--""","""chr14-35115858-35116161-.""",41


Dont forget: there are more sequences in the bin, that are not used in the end, so which are not counted in nr_seqs

In [36]:
multi_prom.filter(pl.col("CRE") == "chr14-20955217-20955500-.")

logFC,nr_barcodes,nr_seqs,label,ENCODE_labels,distance,target_genes,effect,promoter,CRE,promoter_only,z_score,CRE_length,promoter_length,activity_difference
f64,i64,i64,str,str,f64,str,str,str,str,f64,f64,i64,i64,f64
-1.086703,11,2,"""target - other""","""PLS - undefined""",267082.0,"""ANG""","""no effect""","""chr14-20688098-20688455-+""","""chr14-20955217-20955500-.""",-1.379707,0.916286,283,357,4.60108
0.618555,11,1,"""negative - other""","""PLS - undefined""",64362.5,"""RNASE3""","""upregulating""","""chr14-20890827-20891165-+""","""chr14-20955217-20955500-.""",-1.287388,5.517367,283,338,4.60108


In [80]:
multi_prom_with_binorder = actual_bins.select(pl.exclude("column_3")).join(multi_prom, left_on=["left_bin", "right_bin"], right_on=["CRE", "promoter"], coalesce = False)
multi_prom_with_binorder.sort("activity_difference", descending=True).head()


left_bin,right_bin,logFC,nr_barcodes,nr_seqs,label,ENCODE_labels,distance,target_genes,effect,promoter,CRE,promoter_only,z_score,CRE_length,promoter_length,activity_difference
str,str,f64,i64,i64,str,str,f64,str,str,str,str,f64,f64,i64,i64,f64
"""chr14-20955217-20955500-.""","""chr14-20890827-20891165-+""",0.618555,11,1,"""negative - other""","""PLS - undefined""",64362.5,"""RNASE3""","""upregulating""","""chr14-20890827-20891165-+""","""chr14-20955217-20955500-.""",-1.287388,5.517367,283,338,4.60108
"""chr20-17963417-17963734-.""","""chr20-17968096-17968866-+""",-1.259062,8,3,"""target - other""","""PLS - ELS""",4905.5,"""MGME1""","""downregulating""","""chr20-17968096-17968866-+""","""chr20-17963417-17963734-.""",-0.157991,-2.240839,317,770,4.326135
"""chr14-35115858-35116161-.""","""chr14-35121748-35122080--""",0.419135,30,6,"""target - other""","""PLS - undefined""",5904.5,"""PPP2R3C""","""no effect""","""chr14-35121748-35122080--""","""chr14-35115858-35116161-.""",0.046627,1.151496,303,332,3.946451
"""chr15-34095485-34095931-.""","""chr15-34101991-34102521--""",-1.83007,5,1,"""target - other""","""PLS - undefined""",6548.0,"""EMC7""","""downregulating""","""chr15-34101991-34102521--""","""chr15-34095485-34095931-.""",0.189146,-3.633559,446,530,3.777777
"""chr2-99128649-99128811-.""","""chr2-99180852-99181315-+""",1.34942,6,2,"""positive - other""","""PLS - undefined""",52353.5,"""MRPL30""","""no effect""","""chr2-99180852-99181315-+""","""chr2-99128649-99128811-.""",1.026821,0.626273,162,463,3.65902


## Orientation of CREs

In [10]:
pl.concat([silencers_with_bin_order, enhancers_with_bin_order]).select(pl.col("CRE")).write_csv("CREs.withbinorder.temp.ids.tsv", include_header=False)
pl.concat([silencers_with_bin_order, enhancers_with_bin_order]).select(pl.col("promoter")).write_csv("promoter.withbinorder.temp.ids.tsv", include_header=False)

In [ ]:
!awk -v FS="\t" '{print $1"-"$2"-"$3"-"$4"\t"$5"-"$6"-"$7"-"$8"\t"$9}' \
	<(zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/promoteroabinaware_bins_all.filt_OA.bed.gz') \
		| grep -v "\-\-\-" |  grep -Fwf <(paste CREs.withbinorder.temp.ids.tsv promoter.withbinorder.temp.ids.tsv) | \
			> temp.actualbins.withbinorder.tsv

In [ ]:
#### Oke shit het probleem is: dit singlebinssmalleroverlap_OA bestanden werken niet, omdat hier de bin order gesorteerd is ipv de originele bin order.
#### Wacht nee dat klopt niet!!!! Dit is nog voor het sorteren en mergen, dus dit is WEL de originele bin order.

In [91]:
!zcat "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/singlebinssmalleroverlap_left_bins_OA.bed.gz" \
	| grep -Fwf <(sed 's/>//' temp.seqids.tsv) | awk -v OFS="-" '{print $1,$2,$3"\t"$5"\t"$4}'   | grep -Fwf <(cut -f 1 temp.actualbins.withbinorder.tsv | sed 's/-\.//g' | sed 's/-+//g' | sed 's/--//g' ) > left.temp.tsv

In [92]:
!zcat "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/singlebinssmalleroverlap_right_bins_OA.bed.gz" \
	| grep -Fwf <(sed 's/>//' temp.seqids.tsv) | awk -v OFS="-" '{print $1,$2,$3"\t"$5"\t"$4}'   | grep -Fwf <(cut -f 2 temp.actualbins.withbinorder.tsv | sed 's/-\.//g' | sed 's/-+//g' | sed 's/--//g' ) > right.temp.tsv

In [122]:
!join -1 3 -2 3 -o 1.1 2.1 1.2 2.2 1.3 <(sort -k3,3 left.temp.tsv) <(sort -k3,3 right.temp.tsv) -t $'\t' | awk -v OFS="\t" '{print $1,$2,$3,$4,$1"-"$3,$2"-"$4,$5}'\
	| grep -v -Fwf <(cut -f -2 temp.actualbins.withbinorder.tsv | sed 's/-\.//g') | cut -f -6 | sort | uniq -c > temp.promoter.orientation.counts.tsv

In [158]:
!awk '{ \
	key = $2 FS $7; \
	count[key] += $1; \
	row_count[key]++; \
	lines[key,row_count[key]] = $0; \
	val6[key,row_count[key]] = $6; \
	val7[key,row_count[key]] = $7; \
	cnt[key,row_count[key]] = $1 \
} \
END { \
	for (k in count) { \
		if (row_count[k] == 1) { \
			print val6[k,1], val7[k,1] \
		} else if (row_count[k] == 2) { \
			if (cnt[k,1] / count[k] > 0.5) { \
				print val6[k,1], val7[k,1] \
			} else if (cnt[k,2] / count[k] > 0.5) { \
				print val6[k,2], val7[k,2] \
			} else { \
				print "ambiguous", "ambiguous" ; \
			} \
		} \
	} \
} \
' temp.promoter.orientation.counts.tsv > temp.finalbins.tsv # final number of promoters with unique orientation

			# } else { \
			# 	print count[k], lines[k,1], "ambiguous", "ambiguous" ; \
			# 	print count[k], lines[k,2], "ambiguous", "ambiguous" ; \
			# }\

In [24]:
final_bins = pl.read_csv("temp.finalbins.tsv", separator=" ", has_header=False)\
.rename({"column_1": "CRE_withorientation", "column_2": "promoter"}).with_columns(
	CRE = pl.col("CRE_withorientation").str.replace("--","-.") \
		.str.replace("-\+","-."))

enhancers_with_orientation = enhancers_with_bin_order.join(final_bins, on=["promoter", "CRE"]) 
silencers_with_orientation = silencers_with_bin_order.join(final_bins, on=["promoter", "CRE"]) \



## Orientation of CREs multi promoter

In [68]:
all_dual.select(pl.col("CRE")).write_csv("CREs.multiprom.withbinorder.temp.ids.tsv", include_header=False)
all_dual.select(pl.col("promoter")).write_csv("promoter.multiprom.withbinorder.temp.ids.tsv", include_header=False)

In [69]:
!awk -v FS="\t" '{print $1"-"$2"-"$3"-"$4"\t"$5"-"$6"-"$7"-"$8"\t"$9}' \
	<(zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/prommoteroa_bins_all.filt_OA.bed.gz') \
		| grep -v "\-\-\-" | grep -Fwf CREs.multiprom.withbinorder.temp.ids.tsv | grep -Fwf promoter.multiprom.withbinorder.temp.ids.tsv | grep -Fwf <(sed 's/>//' temp.seqids.tsv) \
			> temp.actualbins.multiprom.withbinorder.tsv

In [70]:
!awk '{split($3,seqs,","); for (i in seqs) print $1"\t"$2"\t"seqs[i]}' temp.actualbins.multiprom.withbinorder.tsv | sort | uniq -c | awk '{print $2"\t"$3"\t"$4"\t"$5"\t"$1}' | grep -Fwf <(sed 's/>//' temp.seqids.tsv) > temp.multiprom.orientation.seqs.tsv

In [71]:
!zcat "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/singlebinssmalleroverlap_left_bins_OA.bed.gz" \
	| grep -Fwf <(cut -f 3 temp.multiprom.orientation.seqs.tsv ) | awk -v OFS="-" '{print $1,$2,$3"\t"$5"\t"$4}'   > left.multiprom.temp.tsv

In [72]:
!zcat "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/singlebinssmalleroverlap_right_bins_OA.bed.gz" \
	| grep -Fwf <(cut -f 3 temp.multiprom.orientation.seqs.tsv ) | awk -v OFS="-" '{print $1,$2,$3"\t"$5"\t"$4}'   > right.multiprom.temp.tsv

In [73]:
!join -1 3 -2 3 -o 1.1 2.1 1.2 2.2 1.3 <(sort -k3,3 left.multiprom.temp.tsv) <(sort -k3,3 right.multiprom.temp.tsv) -t $'\t' | awk -v OFS="\t" '{print $1,$2,$3,$4,$1"-"$3,$2"-"$4,$5}'\
 | cut -f -6 | sort | uniq -c > temp.multiprom.part1.promoter.orientation.counts.tsv

In [74]:
!cat temp.multiprom.part1.promoter.orientation.counts.tsv |\
	awk -v OFS="\t" 'NR==FNR {proms[$0]++; next} {if ($6 in proms) print $1,$3,$2,$5,$4,$7,$6; else if ($7 in proms) print $0;}' "promoter.multiprom.withbinorder.temp.ids.tsv" - |\
		awk -v OFS="\t" 'NR==FNR {CREs[$0]++; next} {if ($2"-." in CREs) print $0;}' "CREs.multiprom.withbinorder.temp.ids.tsv" - |\
		awk -v OFS="\t" '{a[$6$7]+=$1; line[$6$7]=$2OFS$3OFS$4OFS$5OFS$6OFS$7} END {for (i in a) print a[i], line[i]}' | sort -k2,2 -k3,3 > \
	temp.multiprom.promoter.orientation.counts.crefirst.tsv 

In [75]:
# Part 1
!awk '{ \
	key = $2 FS $7; \
	count[key] += $1; \
	row_count[key]++; \
	lines[key,row_count[key]] = $0; \
	val2[key,row_count[key]] = $2; \
	val5[key,row_count[key]] = $5; \
	val6[key,row_count[key]] = $6; \
	val7[key,row_count[key]] = $7; \
	cnt[key,row_count[key]] = $1 \
} \
END { \
	for (k in count) { \
		if (row_count[k] == 1) { \
			print cnt[k,1], val6[k,1], val7[k,1] \
		} else if (row_count[k] == 2) { \
			if (cnt[k,1] / count[k] > 0.5) { \
				print cnt[k,1], val6[k,1], val7[k,1] \
			} else if (cnt[k,2] / count[k] > 0.5) { \
				print cnt[k,2], val6[k,2], val7[k,2] \
			} else { \
				print "bboth: "cnt[k,1], val2[k,1]"-"val5[k,1], val7[k,1]"\n",  \
				cnt[k,2], val2[k,2]"-"val5[k,2], val7[k,2]; \
			} \
		} \
	} \
} \
' temp.multiprom.promoter.orientation.counts.crefirst.tsv  > temp.multiprom.finalbins.tsv # final number of promoters with unique orientation

			# } else { \
			# 	print count[k], lines[k,1], "ambiguous", "ambiguous" ; \
			# 	print count[k], lines[k,2], "ambiguous", "ambiguous" ; \
			# }\

In [76]:
!cat temp.multiprom.promoter.orientation.counts.crefirst.tsv | sort -k2,2

1	chr14-20975098-20975210	chr14-20683835-20684397	-	+	chr14-20975098-20975210--	chr14-20683835-20684397-+
1	chr14-20975098-20975210	chr14-20891166-20891415	-	+	chr14-20975098-20975210--	chr14-20891166-20891415-+
1	chr14-34959309-34959415	chr14-34875340-34875553	+	-	chr14-34959309-34959415-+	chr14-34875340-34875553--
1	chr14-34959309-34959415	chr14-34875340-34875553	-	-	chr14-34959309-34959415--	chr14-34875340-34875553--
1	chr14-34959309-34959415	chr14-35122081-35122903	+	-	chr14-34959309-34959415-+	chr14-35122081-35122903--
1	chr14-35441982-35442128	chr14-35122081-35122903	+	-	chr14-35441982-35442128-+	chr14-35122081-35122903--
1	chr14-35441982-35442128	chr14-35404489-35405088	-	-	chr14-35441982-35442128--	chr14-35404489-35405088--
1	chr16-69396489-69396661	chr16-69330001-69330668	+	-	chr16-69396489-69396661-+	chr16-69330001-69330668--
1	chr16-69396489-69396661	chr16-69424473-69424814	-	+	chr16-69396489-69396661--	chr16-69424473-69424814-+
1	chr17-75252859-75253070	chr17-75261404-75261

In [128]:
!cat temp.multiprom.finalbins.tsv | sort -k2,2

bboth: 1 chr14-20955217-20955500-+ chr14-20688098-20688455-+
bboth: 1 chr14-34959309-34959415-- chr14-34875340-34875553--
bboth: 1 chr17-75252859-75253070-- chr17-75261404-75261785--
bboth: 1 chr6-42043482-42043608-+ chr6-41787436-41787975-+
 1 chr14-20955217-20955500-+ chr14-20688098-20688455-+
1 chr14-20955217-20955500-- chr14-20890827-20891165-+
1 chr14-34959309-34959415-+ chr14-35122081-35122903--
 1 chr14-34959309-34959415-- chr14-34875340-34875553--
1 chr14-35441982-35442128-+ chr14-35122081-35122903--
1 chr14-35441982-35442128-- chr14-35404489-35405088--
1 chr16-69396489-69396661-+ chr16-69330001-69330668--
1 chr16-69396489-69396661-- chr16-69424473-69424814-+
 1 chr17-75252859-75253070-- chr17-75261404-75261785--
2 chr17-75252859-75253070-- chr17-75261786-75262174-+
2 chr2-74152117-74152296-+ chr2-74147827-74148390--
1 chr2-74152117-74152296-- chr2-73828674-73829059-+
1 chr20-45465166-45465284-- chr20-45415817-45416355-+
1 chr20-45465166-45465284-- chr20-45631225-45631710--
 1 

## Selecting enhancers and silencers

In [36]:
final_enhancers = enhancers_with_orientation#.join(all_dual, on=["CRE", "promoter", "z_score"], how='anti').filter((pl.col("z_score") > 2))
final_silencers = silencers_with_orientation#.join(all_dual, on=["CRE", "promoter", "z_score"], how='anti').filter((pl.col("z_score") < -2))

## chr20-31604710-31604830-+ does not contain a TSS

final_enhancers = pl.concat([
	final_enhancers.filter((pl.col("promoter") != "chr20-31604710-31604830-+") 
	& (pl.col("promoter") != "chr14-20890827-20891165-+")
	& (pl.col("ENCODE_labels").str.contains("ELS").not_())).sort("z_score", descending=True).head(10),
	final_enhancers.filter((pl.col("promoter") != "chr20-31604710-31604830-+") 
	& (pl.col("promoter") != "chr14-20890827-20891165-+")
	& (pl.col("ENCODE_labels").str.contains("ELS"))).sort("z_score", descending=True).head(10)
])

final_silencers = pl.concat([
	final_silencers.filter(pl.col("ENCODE_labels").str.contains("ELS").not_()).sort("z_score").head(10),
	final_silencers.filter(pl.col("ENCODE_labels").str.contains("ELS")).sort("z_score").head(10)
])

## Selecting different sequence lengths

In [ ]:
high_cov_enh = final_enhancers.sort(["nr_seqs", "z_score"], descending=True).head(3)
high_cov_sil = final_silencers.sort(["nr_seqs"], descending=True).head(3)
low_cov_enh = final_enhancers.sort(["nr_seqs").head(3)

In [9]:
final_enhancers.sort("nr_seqs").head(3)

NameError: name 'final_enhancers' is not defined

In [82]:
high_cov_sil.head()

left_bin,right_bin,logFC,nr_barcodes,nr_seqs,label,ENCODE_labels,distance,target_genes,effect,promoter,CRE,promoter_only,z_score,CRE_length,promoter_length,CRE_withorientation
str,str,f64,i64,i64,str,str,f64,str,str,str,str,f64,f64,i64,i64,str
"""chr1-44988261-44988729-.""","""chr1-45012148-45012412-+""",-0.708816,15,3,"""positive - other""","""PLS - undefined""",23785.0,"""UROD""","""downregulating""","""chr1-45012148-45012412-+""","""chr1-44988261-44988729-.""",0.655515,-3.764063,468,264,"""chr1-44988261-44988729--"""
"""chr14-23028094-23028833-.""","""chr14-23034803-23035146--""",-1.54176,5,3,"""target - other""","""PLS - ELS""",6511.0,"""PSMB5""","""downregulating""","""chr14-23034803-23035146--""","""chr14-23028094-23028833-.""",0.136056,-3.713642,739,343,"""chr14-23028094-23028833-+"""
"""chr20-35768547-35768902-.""","""chr20-35742180-35742625--""",1.125667,9,3,"""target - other""","""PLS - ELS""",26322.0,"""RBM39""","""downregulating""","""chr20-35742180-35742625--""","""chr20-35768547-35768902-.""",2.328104,-3.129785,355,445,"""chr20-35768547-35768902-+"""


In [104]:
pl.concat([high_cov_enh, high_cov_sil]).select("CRE", "promoter")\
	.write_csv("temp.highcovbins1.tsv", include_header=False, separator="\t")
pl.concat([high_cov_enh, high_cov_sil]).select("promoter", "CRE")\
	.write_csv("temp.highcovbins2.tsv", include_header=False, separator="\t")

In [105]:
!cat "temp.highcovbins1.tsv" "temp.highcovbins2.tsv" > "temp.highcovbins.tsv"

In [106]:
# Careful! Some of these bins are in opposite order, the prommoteroa file has the bins sorted horizontally.

!awk -v FS="\t" '{print $1"-"$2"-"$3"-"$4"S"$5"-"$6"-"$7"-"$8"\t"$9}' \
	<(zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/prommoteroa_bins_merged.filt_OA.bed.gz') \
		| grep -v "\-\-\-" |  grep -Fwf <(sed 's/\t/S/' temp.highcovbins.tsv)  |\
		 sed 's/S/\t/g' \
			> temp.highcovbins.seqids.tsv

In [107]:
!awk '{split($3,seqs,","); for (i in seqs) print $1"\t"$2"\t"seqs[i]}' temp.highcovbins.seqids.tsv | grep -Fwf <(sed 's/>//' temp.seqids.tsv) > temp.highcov.seqids.tsv

In [111]:
print(cd)

/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/


In [6]:
!samtools view -N  <(cut -f 3 temp.highcov.seqids.tsv) -d CO:left /data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/merged_samples.tagged.bam > temp.seqs.sam

In [7]:
!wc -l  temp.seqs.sam

17 temp.seqs.sam


## Convert dual CREs file for converting to fasta

In [11]:
!awk -v FS="\t" '$17 == "rev" {print $0FS$10"-"} $17 == "fwd" {print $0FS$10"+"} $17 == "both" {print $0FS$10"-\n"$0FS$10"+"} NR == 1 {print $0}' \
	"/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/luciferase_design_dual_function_withbinorder_withorientation.tsv" |  \
	sed 's/\.-/-/' | sed 's/\.+/+/' | awk -v FS="\t" '$16 == "both" {print $0FS$18FS$9"\n"$0FS$9FS$18} $16 == "CRE-promoter" {print $0FS$18FS$9} $16 == "promoter-CRE" {print $0FS$9FS$18}' \
	| cat <(head -n 1 "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/luciferase_design_dual_function_withbinorder_withorientation.tsv" | \
	paste - <(echo -e '\tleft_bin\tright_bin\tCRE_withorientation')) - \
	> "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/luciferase_design_dual_function_withbinorder_withorientation_forfasta.tsv"

	# awk -v FS="\t" '$16 == "both" {print $0FS$10FS$9"\n"$0FS$9FS$10} $16 == "CRE-promoter" {print $0FS$10FS$9} $16 == "promoter-CRE" {print $0FS$9FS$10}' \
	# "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/luciferase_design_dual_function_withbinorder_withorientation.tsv" \
	# | cat <(head -n 1 "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/luciferase_design_dual_function_withbinorder_withorientation.tsv" | \
	# paste - <(echo -e '\tleft_bin\tright_bin\tCRE_withorientation')) - | \

In [13]:
dual_with_bin_order_and_orientation = pl.read_csv(os.path.join(cd,"results/luciferase_design/luciferase_design_dual_function_withbinorder_withorientation_forfasta.tsv"), separator="\t")


## Write to files

In [ ]:
final_silencers.write_csv(os.path.join(cd,"results/luciferase_design/luciferase_design_silencers.tsv"), separator="\t")
final_enhancers.write_csv(os.path.join(cd,"results/luciferase_design/luciferase_design_enhancers.tsv"), separator="\t")
#all_dual.write_csv(os.path.join(cd,"results/luciferase_design/luciferase_design_dual_function.tsv"), separator="\t")

In [31]:
final_silencers = pl.read_csv(os.path.join(cd,"results/luciferase_design/luciferase_design_silencers.tsv"), separator="\t")
final_enhancers = pl.read_csv(os.path.join(cd,"results/luciferase_design/luciferase_design_enhancers.tsv"), separator="\t")


In [32]:
enhancers_CREs_bed = os.path.join(cd,"results/luciferase_design/coordinates/luciferase_design_enhancers_CREs.bed")
silencers_CREs_bed = os.path.join(cd,"results/luciferase_design/coordinates/luciferase_design_silencers_CREs.bed")
enhancers_promoters_bed = os.path.join(cd,"results/luciferase_design/coordinates/luciferase_design_enhancers_promoters.bed")
silencers_promoters_bed = os.path.join(cd,"results/luciferase_design/coordinates/luciferase_design_silencers_promoters.bed")

In [33]:
final_enhancers.select(pl.col("CRE_withorientation").str.replace_all("-", "\t").str.replace("\t\t", "\t-"), id =
					    pl.col("CRE_withorientation") + "_" + pl.col("promoter")).write_csv(enhancers_CREs_bed,
						 separator="\t", include_header=False, quote_style='never')
final_enhancers.select(pl.col("promoter").str.replace_all("-", "\t").str.replace("\t\t", "\t-"), id =
					    pl.col("CRE_withorientation") + "_" + pl.col("promoter")).write_csv(enhancers_promoters_bed,
						 separator="\t", include_header=False, quote_style='never')

final_silencers.select(pl.col("CRE_withorientation").str.replace_all("-", "\t").str.replace("\t\t", "\t-"), id =
					    pl.col("CRE_withorientation") + "_" + pl.col("promoter")).write_csv(silencers_CREs_bed,
						 separator="\t", include_header=False, quote_style='never')
final_silencers.select(pl.col("promoter").str.replace_all("-", "\t").str.replace("\t\t", "\t-"), id =
					    pl.col("CRE_withorientation") + "_" + pl.col("promoter")).write_csv(silencers_promoters_bed,
						 separator="\t", include_header=False, quote_style='never')

In [15]:
dual_with_bin_order_and_orientation.select(pl.col("left_bin").str.replace_all("-", "\t").str.replace("\t\t", "\t-"), id =
					    pl.col("left_bin") + "_" + pl.col("right_bin")).write_csv(os.path.join(cd,"results/luciferase_design/coordinates/luciferase_design_dual_function_left.bed"),
						 separator="\t", include_header=False, quote_style='never')

dual_with_bin_order_and_orientation.select(pl.col("right_bin").str.replace_all("-", "\t").str.replace("\t\t", "\t-"), id =
					    pl.col("left_bin") + "_" + pl.col("right_bin")).write_csv(os.path.join(cd,"results/luciferase_design/coordinates/luciferase_design_dual_function_right.bed"),
						 separator="\t", include_header=False, quote_style='never')

dual_with_bin_order_and_orientation.select(pl.col("promoter").str.replace_all("-", "\t").str.replace("\t\t", "\t-"), id =
					    pl.col("CRE_withorientation") + "_" + pl.col("promoter")).write_csv(os.path.join(cd,"results/luciferase_design/coordinates/luciferase_design_dual_function_promoters.bed"),
						 separator="\t", include_header=False, quote_style='never')

In [35]:
!awk -v OFS="\t" '{print $1,$2,$3,$5,".",$4}' "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_enhancers_CREs.bed" > temp1 && mv temp1 "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_enhancers_CREs.bed"
!awk -v OFS="\t" '{print $1,$2,$3,$5,".",$4}' "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_silencers_CREs.bed" > temp2 && mv temp2 "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_silencers_CREs.bed"
!awk -v OFS="\t" '{print $1,$2,$3,$5,".",$4}' "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_enhancers_promoters.bed" > temp3 && mv temp3 "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_enhancers_promoters.bed"
!awk -v OFS="\t" '{print $1,$2,$3,$5,".",$4}' "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_silencers_promoters.bed" > temp4 && mv temp4 "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_silencers_promoters.bed"

In [16]:
!awk -v OFS="\t" '{print $1,$2,$3,$5,".",$4}' "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_dual_function_left.bed" > temp1 && mv temp1 "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_dual_function_left.bed"
!awk -v OFS="\t" '{print $1,$2,$3,$5,".",$4}' "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_dual_function_right.bed" > temp1 && mv temp1 "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_dual_function_right.bed"
!awk -v OFS="\t" '{print $1,$2,$3,$5,".",$4}' "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_dual_function_promoters.bed" > temp1 && mv temp1 "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_dual_function_promoters.bed"


In [37]:
!bedtools getfasta -fi "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/resources/reference/hg38.fa" \
	-bed "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_enhancers_CREs.bed" -nameOnly -s \
		-fo "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_enhancers_CREs.fa"
!bedtools getfasta -fi "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/resources/reference/hg38.fa" \
	-bed "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_silencers_CREs.bed" -nameOnly -s \
		-fo "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_silencers_CREs.fa"
!bedtools getfasta -fi "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/resources/reference/hg38.fa" \
	-bed "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_enhancers_promoters.bed" -nameOnly -s \
		-fo "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_enhancers_promoters.fa"
!bedtools getfasta -fi "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/resources/reference/hg38.fa" \
	-bed "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_silencers_promoters.bed" -nameOnly -s \
		-fo "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_silencers_promoters.fa"

In [17]:
!bedtools getfasta -fi "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/resources/reference/hg38.fa" \
	-bed "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_dual_function_left.bed" -nameOnly -s \
		-fo "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_dual_function_left.fa"
!bedtools getfasta -fi "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/resources/reference/hg38.fa" \
	-bed "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_dual_function_right.bed" -nameOnly -s \
		-fo "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_dual_function_right.fa"
!bedtools getfasta -fi "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/resources/reference/hg38.fa" \
	-bed "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_dual_function_promoters.bed" -nameOnly -s \
		-fo "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_dual_function_promoters.fa"

In [39]:
!sed -E '/^>/!{{s/^ATC//I;s/GAT$//I}}' "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_enhancers_CREs.fa" > "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_enhancers_CREs.trimmed.fa" 
!sed -E '/^>/!{{s/^ATC//I;s/GAT$//I}}' "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_silencers_CREs.fa" > "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_silencers_CREs.trimmed.fa"
!sed -E '/^>/!{{s/^ATC//I;s/GAT$//I}}' "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_enhancers_promoters.fa" > "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_enhancers_promoters.trimmed.fa"
!sed -E '/^>/!{{s/^ATC//I;s/GAT$//I}}' "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_silencers_promoters.fa" > "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_silencers_promoters.trimmed.fa"

In [18]:
!sed -E '/^>/!{{s/^ATC//I;s/GAT$//I}}' "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_dual_function_left.fa" > "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_dual_function_left.trimmed.fa"
!sed -E '/^>/!{{s/^ATC//I;s/GAT$//I}}' "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_dual_function_right.fa" > "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_dual_function_right.trimmed.fa"
!sed -E '/^>/!{{s/^ATC//I;s/GAT$//I}}' "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_dual_function_promoters.fa" > "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_dual_function_promoters.trimmed.fa"


In [41]:
!paste -d "~" "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_enhancers_CREs.trimmed.fa" \
	"/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_enhancers_promoters.trimmed.fa"  \
		| sed 's/^>.*~//' | sed 's/([\+-])$//' | sed 's/~/GATCGATC/' \
			> "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_enhancers.fa"

!paste -d "~" "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_silencers_CREs.trimmed.fa" \
	"/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_silencers_promoters.trimmed.fa"  \
		| sed 's/^>.*~//' | sed 's/([\+-])$//' | sed 's/~/GATCGATC/' \
			> "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_silencers.fa"

In [19]:
!paste -d "~" "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_dual_function_left.trimmed.fa" \
	"/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_dual_function_right.trimmed.fa"  \
		| sed 's/^>.*~//' | sed 's/([\+-])$//' | sed 's/~/GATCGATC/' \
			> "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/sequences/luciferase_design_dual_function.fa"

## Check whether TSS is in promoter or in CRE part

In [40]:
!bedtools intersect -a "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_silencers_promoters.bed" \
	-b "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/resources/TSS_pos_v44.bed.gz" -v


In [43]:
!bedtools intersect -a "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_enhancers_promoters.bed" \
	-b "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/resources/TSS_pos_v44.bed.gz" -v


In [83]:
!bedtools intersect -a "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_dual_function_promoters.bed" \
	-b "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/resources/TSS_pos_v44.bed.gz" -v

In [54]:
!bedtools intersect -a <(echo -e "chr14\t34875554\t34876314") \
	-b "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/resources/TSS_pos_v44.bed.gz" 

chr14	34875646	34875647


In [18]:
!sort  "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/luciferase_design/coordinates/luciferase_design_silencers_promoters.bed" | cut -f -3 | uniq -c

      1 chr1	153634006	153634153
      1 chr1	45012148	45012412
      1 chr12	55817310	55818097
      1 chr14	103921385	103921989
      1 chr14	23034803	23035146
      1 chr14	96502122	96502521
      1 chr17	42832923	42833607
      1 chr17	58417524	58417783
      1 chr2	127811101	127811240
      1 chr2	39437371	39438100
      1 chr2	73828674	73829059
      1 chr20	31723132	31724030
      1 chr20	35742180	35742625
      1 chr20	45415817	45416355
      2 chr20	50958049	50958627
      1 chr21	29073313	29073786
      1 chr21	44339303	44339916
      1 chr6	7389289	7389836
      1 chr8	98045434	98045611
